In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor
import warnings
from cluster_based_model import ClusterBasedForecastingModel  # 기존 클러스터 모델 import
warnings.filterwarnings('ignore')

class HybridClusteringStackingModel:
    """클러스터링 + 스택 앙상블 통합 모델"""
    
    def __init__(self):
        # 1. 클러스터링 컴포넌트 (기존 성공한 방식)
        self.cluster_model = None  # 기존 ClusterBasedForecastingModel
        
        # 2. 스택 앙상블 베이스 모델들
        self.base_models = {
            'cluster_based': None,  # 클러스터 모델 결과를 베이스 모델로 사용
            'lgb_conservative': LGBMRegressor(
                n_estimators=400, max_depth=6, learning_rate=0.05,
                feature_fraction=0.8, bagging_fraction=0.8, random_state=42, verbosity=-1
            ),
            'lgb_aggressive': LGBMRegressor(
                n_estimators=600, max_depth=10, learning_rate=0.03,
                feature_fraction=0.9, bagging_fraction=0.9, random_state=43, verbosity=-1
            ),
            'xgb_conservative': XGBRegressor(
                n_estimators=300, max_depth=5, learning_rate=0.08,
                subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0
            ),
            'rf_model': RandomForestRegressor(
                n_estimators=200, max_depth=12, min_samples_split=5,
                random_state=42, n_jobs=-1
            )
        }
        
        # 3. 메타 모델
        self.meta_models = {}
        
        # 4. 클러스터 정보 저장
        self.menu_clusters = {}
        self.cluster_features = {}
        
    def initialize_cluster_model(self):
        """기존 클러스터 모델 초기화 (안전성 강화)"""
        if ClusterBasedForecastingModel is not None:
            try:
                self.cluster_model = ClusterBasedForecastingModel()
            except Exception as e:
                print(f"클러스터 모델 초기화 실패: {e}")
                self.cluster_model = None
        else:
            print("클러스터 모델을 사용할 수 없습니다. 기본 모델로 대체합니다.")
            self.cluster_model = None
        
    def create_hybrid_features(self, df, mode='train'):
        """클러스터 정보를 포함한 하이브리드 피처 생성"""
        
        sequences = []
        targets = []
        metadata = []
        
        for (store, menu), group in df.groupby(['store', 'menu']):
            group = group.sort_values('date').reset_index(drop=True)
            
            seq_length = 28
            min_length = seq_length + (7 if mode == 'train' else 0)
            if len(group) < min_length:
                continue
            
            # 클러스터 정보 가져오기
            cluster_id = self.get_menu_cluster(store, menu)
            cluster_info = self.cluster_features.get(cluster_id, {})
            
            if mode == 'predict':
                seq_data = group.tail(seq_length)
                features = self._create_hybrid_sequence_features(
                    seq_data, store, menu, cluster_id, cluster_info
                )
                sequences.append(features)
                metadata.append({
                    'store': store, 'menu': menu, 'cluster': cluster_id,
                    'cluster_type': cluster_info.get('dominant_type', 'unknown')
                })
                
            else:
                for i in range(len(group) - min_length + 1):
                    seq_data = group.iloc[i:i+seq_length]
                    target_data = group.iloc[i+seq_length:i+seq_length+7]
                    
                    features = self._create_hybrid_sequence_features(
                        seq_data, store, menu, cluster_id, cluster_info
                    )
                    target = target_data['sales'].values
                    
                    sequences.append(features)
                    targets.append(target)
                    metadata.append({
                        'store': store, 'menu': menu, 'cluster': cluster_id,
                        'cluster_type': cluster_info.get('dominant_type', 'unknown')
                    })
        
        X = np.array(sequences) if sequences else np.empty((0, 100))
        y = np.array(targets) if targets else np.empty((0, 7))
        
        return X, y, metadata
    
    def _create_hybrid_sequence_features(self, seq_data, store, menu, cluster_id, cluster_info):
        """클러스터 정보를 포함한 시퀀스 피처 생성"""
        
        sales = seq_data['sales'].values
        features = []
        
        # 1. 기본 통계 피처 (20개)
        features.extend([
            np.mean(sales), np.median(sales), np.std(sales),
            np.min(sales), np.max(sales),
            np.percentile(sales, 25), np.percentile(sales, 75),
            np.mean(sales[-7:]), np.mean(sales[-14:]),
            sales[-1] if len(sales) > 0 else 0,
            (sales == 0).mean(), np.sum(sales > 0), np.sum(sales),
            len(sales), np.var(sales), np.ptp(sales),
            np.mean(np.diff(sales)) if len(sales) > 1 else 0,
            np.std(np.diff(sales)) if len(sales) > 1 else 0,
            np.mean(sales[:7]),
            np.mean(sales[7:14]) if len(sales) > 14 else 0,
        ])
        
        # 2. 클러스터 기반 피처 (15개) - 핵심!
        cluster_features_list = [
            cluster_id if cluster_id != -1 else 0,  # 클러스터 ID
            cluster_info.get('avg_sales', 0),  # 클러스터 평균 매출
            cluster_info.get('zero_ratio', 0),  # 클러스터 0매출 비율
            cluster_info.get('volatility', 0),  # 클러스터 변동성
            cluster_info.get('weekend_effect', 1),  # 클러스터 주말효과
            cluster_info.get('menu_count', 0),  # 클러스터 내 메뉴 수
        ]
        
        # 클러스터 대비 상대적 성과
        cluster_avg = cluster_info.get('avg_sales', np.mean(sales))
        if cluster_avg > 0:
            cluster_features_list.extend([
                np.mean(sales) / cluster_avg,  # 클러스터 평균 대비 현재 성과
                (sales > cluster_avg).mean(),  # 클러스터 평균 초과 비율
                np.std(sales) / cluster_avg,   # 클러스터 대비 상대 변동성
            ])
        else:
            cluster_features_list.extend([1.0, 0.5, 1.0])
        
        # 클러스터 타입 원-핫 인코딩
        cluster_types = ['main', 'drink', 'premium', 'group', 'brunch', 'other']
        cluster_type = cluster_info.get('dominant_type', 'other')
        cluster_features_list.extend([int(cluster_type == ct) for ct in cluster_types])
        
        features.extend(cluster_features_list)
        
        # 3. 라그 및 이동평균 피처 (20개)
        lag_features = []
        for lag in [1, 7, 14, 21]:
            if len(sales) > lag:
                lag_features.append(sales[-lag-1])
                lag_features.append(sales[-1] / (sales[-lag-1] + 1e-8))
            else:
                lag_features.extend([0, 1])
        
        for window in [3, 7, 14]:
            if len(sales) >= window:
                ma = np.mean(sales[-window:])
                lag_features.append(ma)
                lag_features.append(np.std(sales[-window:]))
            else:
                lag_features.extend([0, 0])
        
        features.extend(lag_features[:20])
        
        # 4. 시간 기반 피처 (15개)
        time_features = [
            seq_data['month'].iloc[-1],
            seq_data['day_of_week'].iloc[-1],
            seq_data['is_weekend'].sum(),
            np.sin(2 * np.pi * seq_data['month'].iloc[-1] / 12),
            np.cos(2 * np.pi * seq_data['month'].iloc[-1] / 12),
            np.sin(2 * np.pi * seq_data['day_of_week'].iloc[-1] / 7),
            np.cos(2 * np.pi * seq_data['day_of_week'].iloc[-1] / 7),
        ]
        
        # 요일별 평균
        for dow in range(7):
            dow_data = seq_data[seq_data['day_of_week'] == dow]['sales']
            time_features.append(dow_data.mean() if len(dow_data) > 0 else 0)
        
        features.extend(time_features)
        
        # 5. 업장 원-핫 인코딩 (9개)
        stores = ['담하', '미라시아', '포레스트릿', '카페테리아', '화담숲주막',
                  '화담숲카페', '느티나무 셀프BBQ', '연회장', '라그로타']
        store_features = [int(store == s) for s in stores]
        features.extend(store_features)
        
        # 6. 메뉴 특성 피처 (10개)
        menu_lower = str(menu).lower()
        menu_features = [
            int(any(x in menu_lower for x in ['불고기', '갈비', '찌개', '국밥'])),
            int(any(x in menu_lower for x in ['콜라', '맥주', '소주', '커피'])),
            int(any(x in menu_lower for x in ['정식', '세트', '패키지'])),
            int('브런치' in menu_lower),
            int('단체' in menu_lower),
            int(any(x in menu_lower for x in ['한우', 'aus', '프리미엄'])),
            int(any(x in menu_lower for x in ['아메리카노', '라떼'])),
            int(any(x in menu_lower for x in ['디저트', '아이스크림'])),
            len(menu),
            int('(' in menu)
        ]
        features.extend(menu_features)
        
        # 7. 고급 통계 피처 (11개)
        advanced_features = []
        if len(sales) > 1:
            cv = np.std(sales) / (np.mean(sales) + 1e-8)
            advanced_features.append(cv)
            
            mean_val = np.mean(sales)
            std_val = np.std(sales)
            if std_val > 0:
                skewness = np.mean(((sales - mean_val) / std_val) ** 3)
                kurtosis = np.mean(((sales - mean_val) / std_val) ** 4) - 3
            else:
                skewness = kurtosis = 0
            advanced_features.extend([skewness, kurtosis])
            
            x = np.arange(len(sales))
            trend = np.polyfit(x, sales, 1)[0]
            advanced_features.append(trend)
            
            if len(sales) >= 14:
                momentum = np.mean(sales[-7:]) - np.mean(sales[-14:-7])
            else:
                momentum = 0
            advanced_features.append(momentum)
            
            recent_volatility = np.std(sales[-7:]) if len(sales) >= 7 else 0
            advanced_features.append(recent_volatility)
            
            # 클러스터 정보와 결합한 추가 피처들
            if cluster_id != -1:
                cluster_trend_diff = trend - cluster_info.get('trend_strength', 0)
                cluster_volatility_ratio = cv / (cluster_info.get('volatility', 1) + 1e-8)
                cluster_momentum_diff = momentum - cluster_info.get('momentum_avg', 0)
                performance_vs_cluster = np.mean(sales) / (cluster_info.get('avg_sales', 1) + 1e-8)
                consistency_score = 1 - np.abs(cluster_volatility_ratio - 1)
            else:
                cluster_trend_diff = cluster_volatility_ratio = cluster_momentum_diff = performance_vs_cluster = consistency_score = 0
            
            advanced_features.extend([
                cluster_trend_diff, cluster_volatility_ratio, cluster_momentum_diff,
                performance_vs_cluster, consistency_score
            ])
        else:
            advanced_features = [0] * 11
        
        features.extend(advanced_features)
        
        # NaN 처리
        features = [0.0 if pd.isna(x) or np.isinf(x) else float(x) for x in features]
        
        # 100개 피처로 맞추기
        while len(features) < 100:
            features.append(0.0)
        features = features[:100]
        
        return features
    
    def get_menu_cluster(self, store, menu):
        """메뉴의 클러스터 ID 반환"""
        if hasattr(self.cluster_model, 'menu_clusters'):
            for cluster_id, menus in self.cluster_model.menu_clusters.items():
                if (store, menu) in menus:
                    return cluster_id
        return -1
    
    def fit_with_stacking(self, X, y, metadata):
        """클러스터링 결과를 포함한 스택 앙상블 학습"""
        
        print("하이브리드 스택 앙상블 학습 시작...")
        
        # 1. 클러스터 기반 예측을 베이스 모델 중 하나로 사용
        print("1/4: 클러스터 모델 예측 생성...")
        cluster_predictions = self._get_cluster_base_predictions(X, y, metadata)
        
        # 2. 다른 베이스 모델들로 교차검증 예측 생성
        print("2/4: 다른 베이스 모델들 교차검증...")
        oof_predictions = self._create_stacking_features_with_cluster(X, y, cluster_predictions)
        
        # 3. 메타 모델 학습
        print("3/4: 메타 모델 학습...")
        self._fit_meta_models(oof_predictions, y)
        
        # 4. 전체 데이터로 베이스 모델 재학습
        print("4/4: 베이스 모델 전체 재학습...")
        self._fit_base_models_full(X, y)
        
        print("하이브리드 스택 앙상블 학습 완료!")
    def _get_cluster_base_predictions(self, X, y, metadata):
        """클러스터 모델을 직접 사용한 예측"""
        
        # 메타데이터에서 테스트 데이터 정보 추출
        test_data_info = []
        for meta in metadata:
            test_data_info.append({
                'store': meta['store'],
                'menu': meta['menu'],
                'cluster': meta['cluster']
            })
        
        # 클러스터 모델이 학습된 원본 데이터 형태로 변환
        # (실제로는 테스트 데이터를 클러스터 모델 형태로 재구성해야 함)
        
        # 간단한 방법: 클러스터 모델의 predict 메서드 결과를 직접 사용
        if hasattr(self, '_cached_cluster_predictions'):
            return self._cached_cluster_predictions
        
        # 또는 클러스터 모델을 다시 실행해서 예측값 생성
        # 이미 학습된 클러스터 모델이므로 predict만 호출하면 됨
        
        return self._get_fallback_cluster_predictions(X, metadata)
    
    def _create_stacking_features_with_cluster(self, X, y, cluster_predictions):
        """클러스터 예측을 포함한 스택 피처 생성"""
        
        n_samples = X.shape[0]
        n_base_models = len(self.base_models) - 1  # cluster_based 제외
        
        # OOF 예측값 저장 (클러스터 예측 + 다른 모델들)
        oof_preds = np.zeros((n_samples, (n_base_models + 1) * 7))
        
        # 클러스터 예측을 첫 번째 베이스로 사용
        oof_preds[:, :7] = cluster_predictions
        
        # TimeSeriesSplit으로 다른 모델들 교차검증
        tscv = TimeSeriesSplit(n_splits=3)
        
        model_idx = 1  # 0번은 클러스터 모델
        for model_name, model in self.base_models.items():
            if model_name == 'cluster_based':
                continue
                
            print(f"  교차검증: {model_name}")
            
            for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
                X_train, X_val = X[train_idx], X[val_idx]
                y_train, y_val = y[train_idx], y[val_idx]
                
                multi_model = MultiOutputRegressor(model)
                multi_model.fit(X_train, y_train)
                
                val_pred = multi_model.predict(X_val)
                
                for day in range(7):
                    oof_preds[val_idx, model_idx * 7 + day] = val_pred[:, day]
            
            model_idx += 1
        
        return oof_preds
    
    def _fit_meta_models(self, oof_predictions, y):
        """메타 모델 학습"""
        
        for day in range(7):
            print(f"  메타 모델 학습: Day {day+1}")
            
            y_day = y[:, day]
            X_day = oof_predictions[:, day::7]  # 해당 일의 모든 베이스 예측
            
            # 추가 피처: 전체 통계
            X_day_extended = np.column_stack([
                X_day,
                oof_predictions.mean(axis=1),
                oof_predictions.std(axis=1),
                oof_predictions[:, :7].mean(axis=1),  # 클러스터 모델 평균
            ])
            
            meta_model = Ridge(alpha=0.5)
            meta_model.fit(X_day_extended, y_day)
            self.meta_models[day] = meta_model
    
    def _fit_base_models_full(self, X, y):
        """전체 데이터로 베이스 모델 재학습"""
        
        self.trained_base_models = {}
        
        for model_name, model in self.base_models.items():
            if model_name == 'cluster_based':
                continue
                
            print(f"  전체 학습: {model_name}")
            multi_model = MultiOutputRegressor(model)
            multi_model.fit(X, y)
            self.trained_base_models[model_name] = multi_model
    
    def fit(self, train_df):
        """전체 학습 파이프라인"""
        
        # 1. 클러스터 모델 학습
        print("1/3: 클러스터 모델 학습...")
        self.initialize_cluster_model()
        self.cluster_model.fit(train_df)
        
        # 클러스터 정보 복사
        if hasattr(self.cluster_model, 'menu_clusters'):
            self.menu_clusters = self.cluster_model.menu_clusters
            self.cluster_features = self.cluster_model.cluster_features
        
        # 2. 하이브리드 피처 생성
        print("2/3: 하이브리드 피처 생성...")
        X, y, metadata = self.create_hybrid_features(train_df, mode='train')
        
        if len(X) == 0:
            print("학습 데이터 부족")
            
    
        # 4. 스택 앙상블 학습
        print("3/3: 스택 앙상블 학습...")
        self.fit_with_stacking(X, y, metadata)

    def _generate_cluster_oof_predictions(self, X, y, metadata, train_df):
        """클러스터 모델의 Out-of-Fold 예측 생성"""
        
        from sklearn.model_selection import TimeSeriesSplit
        
        n_samples = len(X)
        oof_cluster_predictions = np.zeros((n_samples, 7))
        
        # TimeSeriesSplit으로 교차검증
        tscv = TimeSeriesSplit(n_splits=3)
        
        print("  클러스터 OOF 예측 생성 중...")
        
        # 메타데이터를 기반으로 원본 데이터 재구성
        reconstructed_data = []
        for i, meta in enumerate(metadata):
            store = meta['store']
            menu = meta['menu']
            
            # 해당 메뉴의 원본 데이터 찾기
            menu_data = train_df[(train_df['store'] == store) & (train_df['menu'] == menu)]
            if len(menu_data) >= 35:  # 충분한 데이터가 있는 경우만
                reconstructed_data.append((i, menu_data))
        
        # 교차검증으로 OOF 예측 생성
        for fold, (train_idx, val_idx) in enumerate(tscv.split(range(len(reconstructed_data)))):
            print(f"    Fold {fold+1}/3")
            
            # 각 폴드별로 클러스터 모델 재학습
            fold_train_data = []
            fold_val_data = []
            fold_val_indices = []
            
            for idx in train_idx:
                original_idx, menu_data = reconstructed_data[idx]
                fold_train_data.append(menu_data)
            
            for idx in val_idx:
                original_idx, menu_data = reconstructed_data[idx]
                fold_val_data.append(menu_data)
                fold_val_indices.append(original_idx)
            
            if fold_train_data and fold_val_data:
                # 폴드 훈련 데이터로 클러스터 모델 재학습
                fold_train_df = pd.concat(fold_train_data, ignore_index=True)
                fold_cluster_model = ClusterBasedForecastingModel()
                fold_cluster_model.fit(fold_train_df)
                
                # 검증 데이터로 예측
                fold_val_df = pd.concat(fold_val_data, ignore_index=True)
                fold_predictions, _ = fold_cluster_model.predict(fold_val_df)
                
                # OOF 예측값 저장
                for i, pred in enumerate(fold_predictions):
                    if i < len(fold_val_indices):
                        oof_cluster_predictions[fold_val_indices[i]] = pred
        
        print(f"  OOF 예측 완료: {oof_cluster_predictions.shape}")
        return oof_cluster_predictions

    def _get_cluster_base_predictions(self, X, y, metadata):
        """저장된 클러스터 예측 결과 사용"""
        
        # 학습 시에는 미리 생성한 OOF 예측 사용
        if hasattr(self, 'cached_cluster_predictions') and y is not None:
            print("  저장된 클러스터 OOF 예측 사용")
            return self.cached_cluster_predictions
        
        # 예측 시에는 실제 클러스터 모델로 예측
        else:
            print("  실시간 클러스터 예측 수행")
            return self._real_time_cluster_prediction(X, metadata)

    def _real_time_cluster_prediction(self, X, metadata):
        """실시간 클러스터 예측 (테스트 시 사용)"""
        
        n_samples = len(X)
        cluster_predictions = np.zeros((n_samples, 7))
        
        # 하이브리드 피처를 클러스터 피처로 변환
        X_cluster = self._convert_hybrid_to_cluster_features(X, metadata)
        
        try:
            # 클러스터 모델의 예측 로직 직접 사용
            has_sales_prob = self.cluster_model.zero_classifier.predict_proba(X_cluster)[:, 1]
            
            for i in range(len(X_cluster)):
                cluster_id = metadata[i]['cluster']
                
                if cluster_id in self.cluster_model.cluster_models:
                    cluster_pred = self.cluster_model.cluster_models[cluster_id].predict(X_cluster[i:i+1])
                    global_pred = self.cluster_model.global_model.predict(X_cluster[i:i+1])
                    cluster_weight = 0.7
                    cluster_predictions[i] = cluster_weight * cluster_pred[0] + (1-cluster_weight) * global_pred[0]
                else:
                    cluster_predictions[i] = self.cluster_model.global_model.predict(X_cluster[i:i+1])[0]
            
            # Zero-inflation 및 후처리
            threshold = 0.3
            zero_mask = has_sales_prob < threshold
            cluster_predictions[zero_mask] = 0
            cluster_predictions = np.maximum(cluster_predictions, 1.0)
            
            return cluster_predictions
            
        except Exception as e:
            print(f"  클러스터 예측 실패: {e}")
            return self._get_fallback_cluster_predictions(X, metadata)

    def _convert_hybrid_to_cluster_features(self, X_hybrid, metadata):
        """하이브리드 피처를 클러스터 피처로 변환"""
        
        # 간단한 매핑 방식 (주요 피처들만 추출)
        X_cluster = np.zeros((len(X_hybrid), 60))
        
        for i in range(len(X_hybrid)):
            # 기본 통계: 하이브리드 0:20 -> 클러스터 0:10
            X_cluster[i, :10] = X_hybrid[i, :10]
            
            # 클러스터 피처: 하이브리드 20:35 -> 클러스터 10:25
            X_cluster[i, 10:25] = X_hybrid[i, 20:35]
            
            # 시간 피처: 하이브리드 44:59 -> 클러스터 25:40
            X_cluster[i, 25:40] = X_hybrid[i, 44:59]
            
            # 업장 피처: 하이브리드 59:68 -> 클러스터 40:49
            X_cluster[i, 40:49] = X_hybrid[i, 59:68]
            
            # 나머지는 0으로 채움
            X_cluster[i, 49:] = 0
        
        return X_cluster
    def predict(self, X_test, test_metadata):
        """하이브리드 예측 (수정된 버전)"""
        
        print("하이브리드 예측 시작...")
        
        # 1. 테스트 데이터에 대한 클러스터 모델 예측 생성
        print("1/3: 클러스터 베이스 예측 생성...")
        cluster_preds = self._get_cluster_predictions_for_test(X_test, test_metadata)
        
        # 2. 다른 베이스 모델들 예측
        print("2/3: 다른 베이스 모델들 예측...")
        other_base_preds = []
        
        for model_name, model in self.trained_base_models.items():
            print(f"  {model_name} 예측 중...")
            pred = model.predict(X_test)
            other_base_preds.append(pred)
        
        # 3. 모든 베이스 예측을 스택 피처로 결합
        print("3/3: 메타 모델 예측...")
        all_base_preds = [cluster_preds] + other_base_preds
        X_stack = np.hstack(all_base_preds)
        
        # 4. 메타 모델로 최종 예측
        final_predictions = np.zeros((X_test.shape[0], 7))
        
        for day in range(7):
            X_day = X_stack[:, day::7]
            X_day_extended = np.column_stack([
                X_day,
                X_stack.mean(axis=1),
                X_stack.std(axis=1),
                X_stack[:, :7].mean(axis=1),  # 클러스터 모델 평균
            ])
            
            day_pred = self.meta_models[day].predict(X_day_extended)
            final_predictions[:, day] = day_pred
        
        return final_predictions
    def _improved_feature_conversion(self, X_hybrid, metadata):
        """하이브리드 피처를 클러스터 피처로 변환 (개선된 버전)"""
        
        X_cluster = np.zeros((len(X_hybrid), 60))
        
        for i in range(len(X_hybrid)):
            # 하이브리드 피처 (100개)에서 클러스터 피처 (60개)로 매핑
            hybrid_features = X_hybrid[i]
            
            # 1. 기본 통계 피처 (0:10)
            basic_features = hybrid_features[:10]
            
            # 2. 클러스터 기반 피처 (20:25) -> (10:15)
            cluster_features = hybrid_features[20:25]
            
            # 3. 클러스터별 상대적 성과 (35:38) -> (15:18)
            relative_features = hybrid_features[35:38]
            
            # 4. 클러스터 타입 (38:43) -> (18:23)
            type_features = hybrid_features[38:43]
            
            # 5. 시간 기반 피처 (44:51) -> (23:30)
            time_features = hybrid_features[44:51]
            
            # 6. 업장 피처 (59:68) -> (30:39)
            store_features = hybrid_features[59:68]
            
            # 7. 라그 피처 (97:100) -> (39:42)
            lag_features = hybrid_features[97:100]
            
            # 클러스터 피처 조합
            cluster_feature_vector = np.concatenate([
                basic_features,      # 10개
                cluster_features,    # 5개  
                relative_features,   # 3개
                type_features,       # 5개
                time_features,       # 7개
                store_features,      # 9개
                lag_features,        # 3개
                np.zeros(18)         # 나머지 18개를 0으로 채움
            ])
            
            X_cluster[i] = cluster_feature_vector[:60]
        
        return X_cluster

    def _get_cluster_predictions_for_test(self, X_test, test_metadata):
        """테스트 데이터에 대한 클러스터 예측 생성 (개선된 버전)"""
        
        print("  클러스터 모델로 테스트 데이터 예측 중...")
        
        # 클러스터 모델이 원하는 형태로 테스트 데이터 재구성
        test_data_for_cluster = []
        
        for i, meta in enumerate(test_metadata):
            store = meta['store']
            menu = meta['menu']
            
            # 하이브리드 피처에서 기본 정보 추출
            # (실제로는 원본 테스트 데이터를 다시 사용하는 것이 더 정확)
            basic_info = {
                'store': store,
                'menu': menu,
                'cluster': meta.get('cluster', -1)
            }
            test_data_for_cluster.append(basic_info)
        
        # 클러스터 모델의 예측 로직을 직접 사용
        try:
            # 하이브리드 피처를 클러스터 피처로 변환
            X_cluster = self._improved_feature_conversion(X_test, test_metadata)
            
            # 클러스터 모델의 예측 파이프라인 실행
            has_sales_prob = self.cluster_model.zero_classifier.predict_proba(X_cluster)[:, 1]
            predictions = np.zeros((len(X_test), 7))
            
            for i in range(len(X_test)):
                cluster_id = test_metadata[i].get('cluster', -1)
                
                if cluster_id in self.cluster_model.cluster_models:
                    cluster_pred = self.cluster_model.cluster_models[cluster_id].predict(X_cluster[i:i+1])
                    global_pred = self.cluster_model.global_model.predict(X_cluster[i:i+1])
                    cluster_weight = 0.7
                    predictions[i] = cluster_weight * cluster_pred[0] + (1-cluster_weight) * global_pred[0]
                else:
                    predictions[i] = self.cluster_model.global_model.predict(X_cluster[i:i+1])[0]
            
            # 클러스터 모델과 동일한 후처리
            threshold = 0.3
            zero_mask = has_sales_prob < threshold
            predictions[zero_mask] = 0
            predictions = np.maximum(predictions, 1.0)
            
            return predictions
        
        except Exception as e:
            print(f"    클러스터 예측 실패: {e}")
            return self._get_fallback_cluster_predictions(X_test, test_metadata)

    def _use_existing_cluster_model(self, X_cluster, test_metadata):
        """기존 학습된 클러스터 모델 사용"""
        
        n_samples = len(X_cluster)
        cluster_predictions = np.zeros((n_samples, 7))
        
        # 매출 여부 분류
        has_sales_prob = self.cluster_model.zero_classifier.predict_proba(X_cluster)[:, 1]
        
        # 메뉴별 예측
        for i in range(n_samples):
            cluster_id = test_metadata[i]['cluster']
            
            if cluster_id in self.cluster_model.cluster_models:
                cluster_pred = self.cluster_model.cluster_models[cluster_id].predict(X_cluster[i:i+1])
                global_pred = self.cluster_model.global_model.predict(X_cluster[i:i+1])
                cluster_weight = 0.7
                cluster_predictions[i] = cluster_weight * cluster_pred[0] + (1-cluster_weight) * global_pred[0]
            else:
                cluster_predictions[i] = self.cluster_model.global_model.predict(X_cluster[i:i+1])[0]
        
        # 후처리 (기존 클러스터 모델과 동일)
        threshold = 0.3
        zero_mask = has_sales_prob < threshold
        cluster_predictions[zero_mask] = 0
        cluster_predictions = np.maximum(cluster_predictions, 1.0)
        
        return cluster_predictions

    def _reconstruct_and_predict_cluster(self, test_metadata):
        """테스트 데이터를 재구성해서 클러스터 모델로 예측"""
        
        print("    테스트 데이터 재구성 방식 사용...")
        
        n_samples = len(test_metadata)
        cluster_predictions = np.zeros((n_samples, 7))
        
        # 각 메뉴별로 기본 예측값 생성
        for i, meta in enumerate(test_metadata):
            store = meta['store']
            menu = meta['menu']
            cluster_id = meta.get('cluster', -1)
            
            # 클러스터 정보 기반 예측
            if cluster_id in self.cluster_features:
                cluster_info = self.cluster_features[cluster_id]
                base_pred = cluster_info.get('avg_sales', 5.0)
                weekend_effect = cluster_info.get('weekend_effect', 1.0)
            else:
                # 업장별 기본값
                store_baselines = {
                    '담하': 8.2, '미라시아': 6.3, '포레스트릿': 47.8,
                    '카페테리아': 18.9, '화담숲주막': 34.4, '화담숲카페': 23.6,
                    '느티나무 셀프BBQ': 5.7, '연회장': 2.3, '라그로타': 1.3
                }
                base_pred = store_baselines.get(store, 5.0)
                weekend_effect = 1.0
            
            # 요일별 패턴 적용
            dow_weights = [0.82, 0.88, 0.91, 0.95, 1.18, 1.34, 1.12]
            adjusted_weights = []
            for j, w in enumerate(dow_weights):
                if j >= 5:  # 주말
                    adjusted_weights.append(w * weekend_effect)
                else:
                    adjusted_weights.append(w)
            
            week_pred = [base_pred * w for w in adjusted_weights]
            
            # 메뉴별 조정
            week_pred = self._apply_menu_adjustments(week_pred, menu)
            
            # 하한값 적용
            week_pred = [max(1.0, p) for p in week_pred]
            
            cluster_predictions[i] = week_pred
        
        return cluster_predictions
    def _get_fallback_cluster_predictions(self, X, metadata):
        """클러스터 모델 사용 불가시 폴백 예측"""
        
        n_samples = len(X)
        fallback_predictions = np.zeros((n_samples, 7))
        
        for i, meta in enumerate(metadata):
            store = meta['store']
            menu = meta['menu']
            cluster_id = meta.get('cluster', -1)
            
            # 업장별 기본값
            store_baselines = {
                '담하': 8.2, '미라시아': 6.3, '포레스트릿': 47.8,
                '카페테리아': 18.9, '화담숲주막': 34.4, '화담숲카페': 23.6,
                '느티나무 셀프BBQ': 5.7, '연회장': 2.3, '라그로타': 1.3
            }
            
            base_pred = store_baselines.get(store, 5.0)
            
            # 클러스터 정보가 있으면 활용
            if cluster_id in self.cluster_features:
                cluster_avg = self.cluster_features[cluster_id].get('avg_sales', base_pred)
                base_pred = (base_pred + cluster_avg) / 2
            
            # 요일별 가중치 적용
            dow_weights = [0.82, 0.88, 0.91, 0.95, 1.18, 1.34, 1.12]
            week_pred = [base_pred * w for w in dow_weights]
            
            # 메뉴별 조정
            week_pred = self._apply_menu_adjustments(week_pred, menu)
            
            # 하한값 적용
            week_pred = [max(1.0, p) for p in week_pred]
            
            fallback_predictions[i] = week_pred
        
        return fallback_predictions
    def _apply_menu_adjustments(self, week_pred, menu):
        """메뉴별 조정 (기존 로직과 동일)"""
        
        menu_lower = str(menu).lower()
        
        if '브런치' in menu_lower:
            brunch_weights = [1.0, 1.0, 1.0, 1.0, 1.1, 1.4, 1.3]
            week_pred = [p * w for p, w in zip(week_pred, brunch_weights)]
        
        elif any(x in menu_lower for x in ['맥주', '소주', '막걸리']):
            alcohol_weights = [0.8, 0.8, 0.9, 1.1, 1.4, 1.6, 1.3]
            week_pred = [p * w for p, w in zip(week_pred, alcohol_weights)]
        
        elif any(x in menu_lower for x in ['커피', '아메리카노', '라떼']):
            cafe_weights = [1.1, 1.1, 1.0, 1.0, 1.2, 1.4, 1.2]
            week_pred = [p * w for p, w in zip(week_pred, cafe_weights)]
        
        elif '단체' in menu_lower:
            total = sum(week_pred)
            if total > 0:
                event_days = np.random.choice(7, size=min(2, 7), replace=False)
                new_pred = [1.0] * 7
                for day in event_days:
                    new_pred[day] = total / len(event_days)
                week_pred = new_pred
        
        return week_pred
# 실행 함수
def run_hybrid_pipeline():
    """클러스터링 + 스택 앙상블 하이브리드 파이프라인"""
    
    print("하이브리드 클러스터링-스택 앙상블 파이프라인 시작...")
    
    # 훈련 데이터 로드
    train_df = pd.read_csv('./train/train.csv')
    train_df['date'] = pd.to_datetime(train_df['영업일자'])
    train_df[['store', 'menu']] = train_df['영업장명_메뉴명'].str.split('_', expand=True, n=1)
    train_df['sales'] = train_df['매출수량']
    train_df['month'] = train_df['date'].dt.month
    train_df['day_of_week'] = train_df['date'].dt.dayofweek
    train_df['is_weekend'] = train_df['day_of_week'].isin([5, 6])
    
    # 하이브리드 모델 학습
    hybrid_model = HybridClusteringStackingModel()
    hybrid_model.fit(train_df)
    
    # 제출 파일 생성
    submission = pd.read_csv('sample_submission.csv')
    
    import glob
    test_files = sorted(glob.glob('TEST_*.csv'))
    
    for test_idx, test_file in enumerate(test_files):
        print(f"하이브리드 예측: {test_file}")
        
        # 테스트 데이터 전처리
        test_df = pd.read_csv(test_file)
        test_df['date'] = pd.to_datetime(test_df['영업일자'])
        test_df[['store', 'menu']] = test_df['영업장명_메뉴명'].str.split('_', expand=True, n=1)
        test_df['sales'] = test_df['매출수량']
        test_df['month'] = test_df['date'].dt.month
        test_df['day_of_week'] = test_df['date'].dt.dayofweek
        test_df['is_weekend'] = test_df['day_of_week'].isin([5, 6])
        
        # 하이브리드 피처 생성
        X_test, _, test_metadata = hybrid_model.create_hybrid_features(test_df, mode='predict')
        
        if len(X_test) > 0:
            # 하이브리드 예측
            predictions = hybrid_model.predict(X_test, test_metadata)
            
            # 제출 파일 매핑
            map_hybrid_predictions_to_submission(
                submission, test_idx, predictions, test_metadata
            )
    
    return submission

def map_hybrid_predictions_to_submission(submission, test_idx, predictions, metadata):
    """하이브리드 예측 결과를 제출 파일에 매핑"""
    
    test_case = f"TEST_{test_idx:02d}"
    test_rows = submission[submission['영업일자'].str.contains(test_case, na=False)].index.tolist()
    
    if len(test_rows) != 7 or len(predictions) == 0:
        return
    
    numeric_cols = submission.select_dtypes(include=[np.number]).columns
    
    for i, meta in enumerate(metadata):
        if i >= len(predictions):
            break
            
        store = meta['store']
        menu = meta['menu']
        target_col = f"{store}_{menu}"
        
        matching_cols = [col for col in numeric_cols if col == target_col]
        if not matching_cols:
            matching_cols = [col for col in numeric_cols 
                           if col.startswith(f"{store}_") and menu in col]
        
        for day_idx, row_idx in enumerate(test_rows):
            if day_idx < predictions.shape[1]:
                pred_value = max(1.0, predictions[i, day_idx])  # 하한값 1.0 유지
                pred_value = round(pred_value, 2)
                
                for col in matching_cols:
                    submission.loc[row_idx, col] = pred_value

# 실행
hybrid_submission = run_hybrid_pipeline()
hybrid_submission.to_csv('hybrid_clustering_stacking_submission.csv', index=False)
print("hybrid_clustering_stacking_submission.csv 생성 완료!")

하이브리드 클러스터링-스택 앙상블 파이프라인 시작...
1/3: 클러스터 모델 학습...
클러스터 기반 모델 학습 시작...
메뉴 클러스터링 시작...


메뉴 특성 추출: 100%|██████████| 193/193 [00:00<00:00, 1614.21it/s]


클러스터 수: 6


클러스터 특성 분석: 100%|██████████| 6/6 [00:01<00:00,  5.27it/s]


클러스터 0: 14개 메뉴, 평균매출 19.1, 타입: group
클러스터 1: 38개 메뉴, 평균매출 21.2, 타입: other
클러스터 2: 10개 메뉴, 평균매출 8.6, 타입: main
클러스터 3: 6개 메뉴, 평균매출 87.5, 타입: other
클러스터 4: 7개 메뉴, 평균매출 21.1, 타입: brunch
클러스터 5: 118개 메뉴, 평균매출 1.9, 타입: other
클러스터 피처: (96114, 60), 타겟: (96114, 7)


...

클러스터 3 완료: other, 2479개 샘플


클러스터 모델:  83%|████████▎ | 5/6 [00:15<00:03,  3.02s/it]      

클러스터 4 완료: brunch, 3292개 샘플


클러스터 모델: 100%|██████████| 6/6 [00:19<00:00,  3.21s/it]      


클러스터 5 완료: other, 46626개 샘플
전체 모델 + 6개 클러스터 모델 학습 완료
2/3: 하이브리드 피처 생성...
3/3: 스택 앙상블 학습...
하이브리드 스택 앙상블 학습 시작...
1/4: 클러스터 모델 예측 생성...
  실시간 클러스터 예측 수행
2/4: 다른 베이스 모델들 교차검증...
  교차검증: lgb_conservative
  교차검증: lgb_aggressive
  교차검증: xgb_conservative
  교차검증: rf_model
3/4: 메타 모델 학습...
  메타 모델 학습: Day 1
  메타 모델 학습: Day 2
  메타 모델 학습: Day 3
  메타 모델 학습: Day 4
  메타 모델 학습: Day 5
  메타 모델 학습: Day 6
  메타 모델 학습: Day 7
4/4: 베이스 모델 전체 재학습...
  전체 학습: lgb_conservative
  전체 학습: lgb_aggressive
  전체 학습: xgb_conservative
  전체 학습: rf_model
하이브리드 스택 앙상블 학습 완료!
하이브리드 예측: TEST_00.csv
하이브리드 예측 시작...
1/3: 클러스터 베이스 예측 생성...
  클러스터 모델로 테스트 데이터 예측 중...
2/3: 다른 베이스 모델들 예측...
  lgb_conservative 예측 중...
  lgb_aggressive 예측 중...
  xgb_conservative 예측 중...
  rf_model 예측 중...
3/3: 메타 모델 예측...
하이브리드 예측: TEST_01.csv
하이브리드 예측 시작...
1/3: 클러스터 베이스 예측 생성...
  클러스터 모델로 테스트 데이터 예측 중...
2/3: 다른 베이스 모델들 예측...
  lgb_conservative 예측 중...
  lgb_aggressive 예측 중...
  xgb_conservative 예측 중...
  rf_model 예측 중...
3/3: 메타 모델 예측...
하이브리드 예